# Ground Truth Validation with spectral_select

This notebook demonstrates how to validate clustering results against ground truth annotations using the `spectral_select` library.

**What you'll learn:**
- Load ground truth from annotated PNG images
- Compute validation metrics (ARI, NMI, purity, etc.)
- Generate validation reports
- Visualize validation results

## 1. Setup

Import the validation classes from `spectral_select`:

In [ ]:
from spectral_select import Validator, Visualizer, load_ground_truth_from_png
import numpy as np

## 2. Load Ground Truth

Load ground truth labels from an annotated PNG image. The PNG should use distinct colors for each class (background pixels use a separate color).

In [ ]:
# Option A: Auto-detect classes from unique colors
# ground_truth = load_ground_truth_from_png("path/to/annotations.png")

# Option B: Define class colors explicitly for tolerance-based matching
GT_PATH = "../../Data/Processed/Lichens_2/ground_truth.png"

ground_truth = load_ground_truth_from_png(
    GT_PATH,
    class_colors={
        "substrate": (0, 255, 0),      # Green
        "lichen_1": (255, 0, 0),       # Red  
        "lichen_2": (0, 0, 255),       # Blue
    },
    color_tolerance=50,  # Euclidean distance tolerance for matching
)

print(f"Ground truth shape: {ground_truth.labels.shape}")
print(f"Classes: {ground_truth.class_names}")
print(f"Color mapping: {ground_truth.color_mapping}")

## 3. Load Clustering Results

Load your clustering predictions. These should be a 2D array matching the ground truth shape, where each pixel has a cluster label.

In [ ]:
# Load clustering results from saved file
PREDICTIONS_PATH = "../../results/clustering/cluster_labels.npy"

predictions = np.load(PREDICTIONS_PATH)

print(f"Predictions shape: {predictions.shape}")
print(f"Unique clusters: {np.unique(predictions)}")

## 4. Run Validation

Fit the Validator to compute all metrics. The validator handles:
- Optimal cluster-to-GT assignment (using Hungarian algorithm)
- Excluding background pixels (-1) from scoring
- Computing comprehensive metrics

In [ ]:
validator = Validator()
validator.fit(predictions, ground_truth)

# Primary metric: Adjusted Rand Index
print(f"ARI Score: {validator.score():.4f}")

## 5. Explore Metrics

Access all computed metrics through the `metrics` property:

In [ ]:
m = validator.metrics

print("=== Clustering Metrics ===")
print(f"Adjusted Rand Index:     {m.adjusted_rand_index:.4f}")
print(f"Normalized Mutual Info:  {m.normalized_mutual_info:.4f}")
print(f"V-Measure:               {m.v_measure:.4f}")
print(f"Homogeneity:             {m.homogeneity:.4f}")
print(f"Completeness:            {m.completeness:.4f}")
print(f"Fowlkes-Mallows:         {m.fowlkes_mallows:.4f}")
print(f"Purity:                  {m.purity:.4f}")
print()
print(f"Overall Accuracy:        {m.overall_accuracy:.4f}")
print(f"Total Pixels Evaluated:  {m.total_pixels}")

## 6. Per-Class Analysis

Examine performance for each ground truth class:

In [ ]:
print("=== Per-Class Metrics ===")
print(f"{'Class':<15} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
print("-" * 55)

for class_id, metrics in m.per_class_metrics.items():
    class_name = ground_truth.class_names.get(class_id, f"Class {class_id}")
    print(
        f"{class_name:<15} "
        f"{metrics['precision']:>10.4f} "
        f"{metrics['recall']:>10.4f} "
        f"{metrics['f1']:>10.4f} "
        f"{metrics['support']:>10d}"
    )

## 7. Generate Report

Create a comprehensive Markdown-formatted report:

In [ ]:
report = validator.generate_report()
print(report)

Optionally save the report to a file:

In [ ]:
# from pathlib import Path
# validator.generate_report(output_path=Path("validation_report.md"))

## 8. Visualize Results

Create visualizations using `Visualizer.from_validation()`:

In [ ]:
viz = Visualizer.from_validation(validator, predictions)

In [ ]:
# Confusion matrix: rows = true class, columns = predicted cluster
viz.plot_confusion_matrix(
    cm=validator.metrics.confusion_matrix,
    class_names=list(ground_truth.class_names.values()),
)

In [ ]:
# Per-class metrics bar chart
viz.plot_per_class_metrics(validator.metrics.per_class_metrics)

In [ ]:
# Spatial accuracy heatmap
viz.plot_accuracy_heatmap(
    ground_truth=ground_truth.labels,
    predictions=predictions,
)

## Summary

In this notebook, you learned how to:

1. **Load ground truth** with `load_ground_truth_from_png()`
2. **Run validation** with `Validator.fit(predictions, ground_truth)`
3. **Get metrics** with `validator.score()` and `validator.metrics`
4. **Generate reports** with `validator.generate_report()`
5. **Visualize** with `Visualizer.from_validation()`

**Key metrics explained:**
- **ARI** (Adjusted Rand Index): Measures agreement between clusterings, adjusted for chance. Range: [-1, 1], higher is better.
- **NMI** (Normalized Mutual Info): Information-theoretic measure of clustering quality. Range: [0, 1].
- **Purity**: Fraction of majority class in each cluster. Simple but sensitive to cluster count.
- **V-Measure**: Harmonic mean of homogeneity and completeness.

**Next steps:**
- See `01_quickstart.ipynb` for the full wavelength selection workflow
- Try different clustering algorithms on the selected wavelengths
- Compare validation metrics across different band selections